# Lab 06 solution: Add a fourth tool and give it to both hosts

The worked solution to the lab's challenge, run with its outputs. It repeats the setup, the client and the two hosts
from the lab, so it runs on its own. The server copy with the fourth tool is `solved/chinook_server_plus.py`: the lab's
`chinook_server.py` with one more `TypedDict`, `RecentInvoice`, and one more tool, `get_recent_invoices`.

## Setup

In [1]:
from pathlib import Path
import json
import os
import sqlite3
import subprocess
import sys
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


## Task 1: Add the tool and check what the server publishes

The new tool, as it reads in the server copy:

In [2]:
PLUS_FILE = ROOT / "06_MCP" / "solved" / "chinook_server_plus.py"

lines = PLUS_FILE.read_text().splitlines()

# The new shape (4 lines) and the new tool (9 lines, with its decorator)
start = lines.index("class RecentInvoice(TypedDict):")
print("\n".join(lines[start:start + 4]), "\n")
start = lines.index("def get_recent_invoices(customer_id: int, limit: int = 3) -> list[RecentInvoice]:") - 1
print("\n".join(lines[start:start + 9]))

class RecentInvoice(TypedDict):
    invoice_id: int
    date: str
    total: float 

@mcp.tool()
def get_recent_invoices(customer_id: int, limit: int = 3) -> list[RecentInvoice]:
    """List a customer's most recent invoices, newest first, with their date and total in US dollars."""
    rows = con.execute(
        "SELECT InvoiceId, date(InvoiceDate), Total FROM invoices WHERE CustomerId = ? "
        "ORDER BY InvoiceDate DESC LIMIT ?",
        (customer_id, limit),
    ).fetchall()
    return [{"invoice_id": r[0], "date": r[1], "total": r[2]} for r in rows]


`limit: int = 3` has a default, so a caller may leave it out. `date(InvoiceDate)` keeps only the `YYYY-MM-DD` part of
the stored timestamp, and the second `?` in the SQL takes `limit`. We list the copy's tools with the `mcp` client.

In [3]:
from mcp import Client, StdioServerParameters

PLUS = StdioServerParameters(command=sys.executable, args=[str(PLUS_FILE)])

async with Client(PLUS) as mcp_client:
    listed = await mcp_client.list_tools()

print([tool.name for tool in listed.tools])
new_tool = listed.tools[-1]
print(json.dumps(new_tool.input_schema, indent=2))

['find_customer', 'get_invoices', 'get_support_rep', 'get_recent_invoices']
{
  "type": "object",
  "properties": {
    "customer_id": {
      "title": "Customer Id",
      "type": "integer"
    },
    "limit": {
      "default": 3,
      "title": "Limit",
      "type": "integer"
    }
  },
  "required": [
    "customer_id"
  ],
  "title": "get_recent_invoicesArguments"
}


Four tools. In the new tool's `inputSchema`, `limit` has `"default": 3` and only `customer_id` is `required`: the
server wrote the optional parameter from the default in the function's signature. Its `outputSchema` wraps the list in
`result`, as for `find_customer`:

In [4]:
print(json.dumps(new_tool.output_schema["$defs"]["RecentInvoice"]["properties"], indent=2))

{
  "invoice_id": {
    "title": "Invoice Id",
    "type": "integer"
  },
  "date": {
    "title": "Date",
    "type": "string"
  },
  "total": {
    "title": "Total",
    "type": "number"
  }
}


Called by hand for customer 6, with no `limit`:

In [5]:
async with Client(PLUS) as mcp_client:
    result = await mcp_client.call_tool("get_recent_invoices", {"customer_id": 6})

for invoice in result.structured_content["result"]:
    print(invoice)

{'invoice_id': 404, 'date': '2013-11-13', 'total': 25.86}
{'invoice_id': 393, 'date': '2013-10-03', 'total': 1.98}
{'invoice_id': 272, 'date': '2012-04-11', 'total': 0.99}


Three invoices, newest first: 404 (2013-11-13, 25.86), 393 (2013-10-03, 1.98) and 272 (2012-04-11, 0.99). The part a
host sends to the model, `content`, holds them as separate text blocks, one per invoice:

In [6]:
print("text blocks:", len(result.content))
for block in result.content:
    print(block.text.replace("\n", " "))

text blocks: 3
{   "invoice_id": 404,   "date": "2013-11-13",   "total": 25.86 }
{   "invoice_id": 393,   "date": "2013-10-03",   "total": 1.98 }
{   "invoice_id": 272,   "date": "2012-04-11",   "total": 0.99 }


Three blocks. This is why the lab's loop joins every block: sending only `content[0].text` would give the model the
newest invoice and nothing else.

## Task 2: Give it to both hosts without changing their code

The loop and the tool conversion from the lab, unchanged. The only difference is the server it starts: `SERVER` now
names the copy.

In [7]:
from openai import OpenAI, omit

client = OpenAI()
SERVER = PLUS
SYSTEM = "You are a support assistant for the Chinook music shop."


def to_chat_tool(tool):
    """Turn one MCP tool into the tool dict Chat Completions expects."""
    return {
        "type": "function",
        "function": {"name": tool.name, "description": tool.description, "parameters": tool.input_schema},
    }


def ask_model(messages, tools):
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,
        reasoning_effort=EFFORT or omit,
        max_completion_tokens=400,
    )
    return response.choices[0].message

In [8]:
async def run_agent(question, max_turns=5):
    async with Client(SERVER) as mcp_client:
        listed = await mcp_client.list_tools()
        tools = [to_chat_tool(tool) for tool in listed.tools]
        messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
        calls = []
        for turn in range(1, max_turns + 1):
            reply = ask_model(messages, tools)
            if not reply.tool_calls:
                return reply.content, turn, calls
            messages.append(reply)
            for call in reply.tool_calls:
                args = json.loads(call.function.arguments)
                result = await mcp_client.call_tool(call.function.name, args)
                calls.append((turn, call.function.name, args, result.structured_content))
                # Every text block, joined: a list result comes as one block per item
                text = "\n".join(block.text for block in result.content if block.type == "text")
                messages.append({"role": "tool", "tool_call_id": call.id, "content": text})
    return None, max_turns, calls

`max_completion_tokens` is 400 here instead of 300: the answer lists three invoices with their dates and totals. Our
loop first:

In [9]:
QUESTION = "What were Helena Holý's three most recent invoices?"

answer, model_calls, calls = await run_agent(QUESTION)
for turn, name, args, structured in calls:
    print(f"turn {turn}: {name}({args})")
print("model calls:", model_calls)
print(answer)

turn 1: find_customer({'name': 'Helena Holý'})
turn 2: get_recent_invoices({'customer_id': 6, 'limit': 3})
model calls: 3
Helena Holý’s three most recent invoices were:

- **Invoice 404** — November 13, 2013 — **$25.86**
- **Invoice 393** — October 3, 2013 — **$1.98**
- **Invoice 272** — April 11, 2012 — **$0.99**


Three model calls: `find_customer` on turn 1, then `get_recent_invoices` with `customer_id` 6 (the model also sent
`limit` 3, though it could have left it out), then the answer: invoices 404, 393 and 272 with their dates and totals.
The loop's code is the lab's; it found the fourth tool because it asks the server for its tools at the start of every
run.

Now the Agents SDK, with a new `MCPServerStdio` pointed at the copy:

In [10]:
from agents import Agent, ModelSettings, Runner, set_tracing_disabled
from agents.mcp import MCPServerStdio
from openai.types.shared import Reasoning

set_tracing_disabled(True)

plus_server = MCPServerStdio(name="chinook-plus", params={"command": sys.executable, "args": [str(PLUS_FILE)]})
agent = Agent(
    name="Chinook desk",
    instructions=SYSTEM,
    model=MODEL,
    mcp_servers=[plus_server],
    model_settings=ModelSettings(reasoning=Reasoning(effort=EFFORT)),
)

async with plus_server:
    sdk_result = await Runner.run(agent, QUESTION)

for item in sdk_result.new_items:
    if item.type == "tool_call_item":
        print("tool call:", item.raw_item.name, item.raw_item.arguments)
print("model calls:", sdk_result.context_wrapper.usage.requests)
print(sdk_result.final_output)

tool call: find_customer {"name":"Helena Holý"}
tool call: get_recent_invoices {"customer_id":6,"limit":3}
model calls: 3
Helena Holý’s three most recent invoices were:

- **Invoice 404** — November 13, 2013: **$25.86**
- **Invoice 393** — October 3, 2013: **$1.98**
- **Invoice 272** — April 11, 2012: **$0.99**


The same two tool calls (`find_customer`, then `get_recent_invoices` with `customer_id` 6 and `limit` 3), three
model calls, and the same three invoices. The SDK's code did not change either: `MCPServerStdio` listed four tools
because the server now offers four.

A check in code on both replies: the three invoice numbers and their three totals. Dates are left out of the check,
because a reply may write 2013-11-13 as "November 13, 2013"; read them in the replies above.

In [11]:
for host, text in [("our loop", answer), ("Agents SDK", sdk_result.final_output)]:
    found = [fact for fact in ["404", "393", "272", "25.86", "1.98", "0.99"] if fact in text]
    print("✅" if len(found) == 6 else "❌", host, found)

✅ our loop ['404', '393', '272', '25.86', '1.98', '0.99']
✅ Agents SDK ['404', '393', '272', '25.86', '1.98', '0.99']


Both replies hold the three invoice numbers and the three totals. The check reads strings: a reply that gave the
dates and totals without the invoice numbers would fail it while being right, so read the replies too. Neither host's code changed: each asked the server for its tools at the start of the run, found the
fourth one, and the model chose it. That is the point of the protocol: a tool added once, on the server, reaches every
host that connects to it.